In [1]:
# pip installer
%pip install nflreadpy
!pip install pandas
!pip install polars
!pip install sklearn

In [2]:
# Import libraries
import nflreadpy as nfl
import pandas as pd
import polars as pl


from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score


In [3]:
## Load data 
seasons = list(range(2018, 2025)) # Slicing to the last 7-sevens
pbp = nfl.load_pbp(seasons=seasons)


In [4]:
# Showing the first 5-lines
pbp.head()

In [5]:
pbp.shape # Printing the shape of the Polars dataframe

In [6]:
# Getting a better understanding what EPA is
pbp.select("epa").describe()

In [7]:
pbp.select("play_type").unique() # What is inside the play_type column

In [8]:
pbp.select("season_type").unique()
# since there is only post and regular, we keep both of them

In [9]:
# Keeping the plays where EPA is not null
pbp = pbp.filter(
    (pl.col("epa").is_not_null()) &
    (pl.col("play_type").is_in(["run","pass"]))  # removing qb_kneels, field goals and so forth
)

In [10]:
pbp.shape # printing the new shape of the dataframe

In [11]:
pbp.columns

In [12]:
pbp.select("posteam").unique()

In [13]:
# Team-game aggregations from the pbp
## Computing offensive efficiency per team per game

team_game = ( 
    pbp.group_by(["season", "week", "game_id", "posteam"])
    .agg([
        pl.mean("epa").alias("off_epa_per_play"), # storing mean of epa as off_epa_per_play
        (pl.col("epa") > 0).mean().alias("off_success_rate"), # Storing the offensive success rate where the EPA is increased
        (pl.col("play_type") == "pass").mean().alias("off_pass_rate"), # storing the success for play type pass
        (pl.col("play_type") == "run").mean().alias("off_run_ra te"),
        pl.len().alias("off_plays") # number of offensive plays
    ])
    .rename({"posteam":"team"})
)

In [14]:
# Game outcomes (winner lables) from game-level info embeeded in pbp

games = (
    pbp.group_by(["season", "week", "game_id"])
    .agg([
        pl.first("home_team").alias("home_team"),
        pl.first("away_team").alias("away_team"),
        pl.first("home_score").alias("home_score"),
        pl.first("away_score").alias("away_score"),
        pl.first("game_date").alias("game_date")
    ])
    .with_columns([
        (pl.col("home_score") > pl.col("away_score")).cast(pl.Int8).alias("home_win"),
        pl.col("game_date").cast(pl.Date).alias("game_date")
    ])
)

In [15]:
# Creating rolling pre-game features per team

window = 4 # Window is based on the last 4 games

team_game = ( 
    team_game.sort(["team", "season","week"])
    .with_columns([
        pl.col("off_epa_per_play").shift(1).over("team").rolling_mean(window).alias("r_off_epa"),
        pl.col("off_success_rate").shift(1).over("team").rolling_mean(window).alias("r_off_succ"),
        pl.col("off_pass_rate").shift(1).over("team").rolling_mean(window).alias("r_off_pass")
    ])
)

In [16]:
# keep only rows where there is "enough" data 
team_game = team_game.filter(
    pl.all_horizontal([
        pl.col("r_off_epa").is_not_null(),
        pl.col("r_off_succ").is_not_null(),
        pl.col("r_off_pass").is_not_null(),
    ])
)

In [17]:
# Join rolling features for home and away teams into one game row


home_feats = team_game.rename({
    "team": "home_team",
    "r_off_epa": "home_r_off_epa",
    "r_off_succ":"home_r_off_succ",
    "r_off_pass": "home_r_off_pass"
})

away_feats = team_game.rename({
    "team": "away_team",
    "r_off_epa": "away_r_off_epa",
    "r_off_succ":"away_r_off_succ",
    "r_off_pass": "away_r_off_pass"
})



dataset = (
    games.join(home_feats.select(["season", "week", "game_id", "home_team",
                                  "home_r_off_epa", "home_r_off_succ", "home_r_off_pass"]),
                                  on=["season","week", "game_id","home_team"], how="inner")
        .join(away_feats.select(["season", "week", "game_id", "away_team",
                                  "away_r_off_epa", "away_r_off_succ", "away_r_off_pass"]),
               on=["season", "week", "game_id", "away_team"], how="inner")
        .with_columns([
            (pl.col("home_r_off_epa") - pl.col("away_r_off_epa")).alias("diff_r_off_epa"),
            (pl.col("home_r_off_succ") - pl.col("away_r_off_succ")).alias("diff_r_off_succ"),
            (pl.col("home_r_off_pass") - pl.col("away_r_off_pass")).alias("diff_r_off_pass")
        ])       
               
)

# Converting to pandas for sklearn
df = dataset.to_pandas()
    

In [18]:
# feature extraction 

FEATURES = [
    "home_r_off_epa", "away_r_off_epa", "diff_r_off_epa",
    "home_r_off_succ", "away_r_off_succ", "diff_r_off_succ",
    "home_r_off_pass","away_r_off_pass", "diff_r_off_pass"
]

TARGET = "home_win"

In [19]:
# Visual correlation matrix comparision
import seaborn as sns
import matplotlib.pyplot as plt

corr = df[FEATURES + [TARGET]].corr()

sns.heatmap(corr, cmap="coolwarm",center=0)
plt.show()

In [20]:
# Time based splits 

test_season = df["season"].max() # latest/current season

train = df[df["season"] < test_season].copy()
test = df[df["season"] == test_season].copy()


X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

In [21]:
# model
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

In [22]:
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("Test season:", test_season)
print("Accuracy:", accuracy_score(y_test, pred))
print("AUC:", roc_auc_score(y_test, proba))
print("LogLoss:", log_loss(y_test, proba))

In [23]:
# Fine tuning model by adding defensive features + Rest days + Short weeks

In [26]:
# Handling defensive performance
team_def_game = ( 
    pbp.group_by(["season", "week","game_id","defteam"])
    .agg([
        pl.mean("epa").alias("def_epa_allowed_pp"),
        (pl.col("epa") > 0).mean().alias("def_succ_allowed"),
        pl.len().alias("def_plays_faced"),
    ])
    .rename({"defteam":"team"})
)

team_def_game = (
    team_def_game.sort(["team", "season", "week"])
    .with_columns([
        pl.col("def_epa_allowed_pp").shift(1).over("team").rolling_mean(window).alias("r_def_epa_allowed"),
        pl.col("def_succ_allowed").shift(1).over("team").rolling_mean(window).alias("r_def_succ_alloed")
    ])
)


# Hanlding the Rest days and short weeks
team_schedule = (
    games.select(["season", "week", "game_id", "game_date", "home_team", "away_team"])
    .with_columns([
        pl.lit(1).alias("is_home") #placeholder for home rows
    ])   
)


home_rows = team_schedule.select([
    "season", "week", "game_id", "game_date",
    pl.col("home_team").alias("team"),
    pl.lit(1).alias("is_home")
])

away_rows = team_schedule.select([
    "season", "week", "game_id", "game_date",
    pl.col("away_team").alias("team"),
    pl.lit(0).alias("is_home")
])


team_games = pl.concat([home_rows, away_rows]).sort(["team", "season", "week"])

team_games = (
    team_games.with_columns([
        pl.col("game_date").shift(1).over("team").alias("prev_game_date")
    ])
    .with_columns([
        (pl.col("game_date") - pl.col("prev_game_date")).dt.total_days().alias("rest_days")
    ])
    .with_columns([
        (pl.col("rest_days") <= 6).cast(pl.Int8).alias("short_week")
    ])
)

In [27]:
# rest
home_rest = team_games.select(["season", "week","game_id", pl.col("team").alias("home_team"), "rest_days", "short_week"])\
            .rename({"rest_days":"home_rest_days", "short_week":"home_short_week"})

away_rest= team_games.select(["season", "week","game_id", pl.col("team").alias("away_team"), "rest_days", "short_week"])\
            .rename({"rest_days":"away_rest_days", "short_week":"away_short_week"})

In [30]:
# QB rolling features
qb_plays = pbp.filter(
    (pl.col("pass_attemps") == 1) & #binary column
    pl.col("passer_player_id").is_not_null() &
    (pl.col())
    )

In [32]:
pass_players = (
    pbp.group_by(["team","season","week","passer_player_id"])
    .with_columns(pl.col("pass_attempts"))
)

print(pass_players)